### Import & Data Load


In [1]:
# Import & Data Load
# (우리는 이미 로컬 가상환경에 필요한 패키지들을 설치 완료했습니다)

In [2]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
train.head(5)

,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


### Check Data


In [5]:
train.isnull().sum()

ID                             0
gender                         0
age                            0
height                         0
weight                         0
cholesterol                    0
systolic_blood_pressure        0
diastolic_blood_pressure       0
glucose                        0
bone_density                   0
activity                       0
smoke_status                   0
medical_history             1289
family_medical_history      1486
sleep_pattern                  0
edu_level                    607
mean_working                1032
stress_score                   0
dtype: int64

In [6]:
# 결측값 있는 칼럼(column) 확인
missing_columns_train = train.columns[train.isnull().sum() > 0]
missing_columns_train

Index(['medical_history', 'family_medical_history', 'edu_level',
       'mean_working'],
      dtype='object')

In [7]:
train[missing_columns_train].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   medical_history         1711 non-null   object 
 1   family_medical_history  1514 non-null   object 
 2   edu_level               2393 non-null   object 
 3   mean_working            1968 non-null   float64
dtypes: float64(1), object(3)
memory usage: 93.9+ KB


In [8]:
# 결측값이 있는 범주형 변수 탐색 (이전 결측값 수동 처리 완료)

### Data Preprocessing


In [9]:
# 범주형 및 수치형 변수의 결측값을 보다 정밀하게 대체합니다.
# 기존 병력(medical_history) 및 가족력(family_medical_history)의 결측은 특정 병력이 없음을 의미할 가능성이 큽니다.
train['medical_history'] = train['medical_history'].fillna('none')
test['medical_history'] = test['medical_history'].fillna('none')

train['family_medical_history'] = train['family_medical_history'].fillna('none')
test['family_medical_history'] = test['family_medical_history'].fillna('none')

# 학력의 결측값은 unknown으로 대체합니다.
train['edu_level'] = train['edu_level'].fillna('unknown')
test['edu_level'] = test['edu_level'].fillna('unknown')

In [10]:
# mean_working에 대해 중앙값 대체
median_value = train['mean_working'].median()
train['mean_working'] = train['mean_working'].fillna(median_value)
test['mean_working'] = test['mean_working'].fillna(median_value)

# --- Feature Engineering ---
# 1. BMI 지수 생성
for df in [train, test]:
    df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# 2. 맥압(Pulse Pressure) 및 평균혈압(Mean Blood Pressure) 생성
for df in [train, test]:
    df['pulse_pressure'] = df['systolic_blood_pressure'] - df['diastolic_blood_pressure']
    df['mean_pressure'] = df['diastolic_blood_pressure'] + df['pulse_pressure'] / 3

In [11]:
# 범주형 데이터 변환 및 타입 지정
categorical_cols = ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'edu_level', 'sleep_pattern']

for col in categorical_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

In [12]:
x_train = train.drop(['ID', 'stress_score'], axis = 1)
y_train = train['stress_score']

x_test = test.drop('ID', axis = 1)

### Train / Predict


In [13]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(x_train))
test_preds = np.zeros(len(x_test))
rmse_list = []

# LightGBMRegressor 학습 (최적화된 하이퍼파라미터 적용)
for fold, (train_idx, val_idx) in enumerate(kf.split(x_train)):
    X_tr, y_tr = x_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = x_train.iloc[val_idx], y_train.iloc[val_idx]
    
    model = LGBMRegressor(learning_rate=0.1, num_leaves=63, n_estimators=200, random_state=42, verbose=-1)
    model.fit(X_tr, y_tr)
    
    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds
    rmse = root_mean_squared_error(y_val, val_preds)
    rmse_list.append(rmse)
    
    # Fold별 테스트 예측값 누적 평균
    test_preds += model.predict(x_test) / 5
    print(f'Fold {fold+1} RMSE: {rmse:.6f}')

print(f'==> 5-Fold Average RMSE: {np.mean(rmse_list):.6f}')

Fold 1 RMSE: 0.232382


Fold 2 RMSE: 0.243364


Fold 3 RMSE: 0.244271


Fold 4 RMSE: 0.252484


Fold 5 RMSE: 0.246123
==> 5-Fold Average RMSE: 0.243725


### Submission


In [14]:
submission = pd.read_csv('sample_submission.csv')

In [15]:
submission['stress_score'] = test_preds
submission.head()

,ID,stress_score
0,TEST_0000,0.482957
1,TEST_0001,0.792289
2,TEST_0002,0.260243
3,TEST_0003,0.436104
4,TEST_0004,0.554552


In [16]:
submission.to_csv('submit.csv', index=False)